# Week 03 – CartFlow Data Exploration

**Project:** P06 CartFlow — Retail & E-commerce Analytics

**Purpose:** Profile the five approved batch sources, confirm grains and keys, test relationships, and quantify the one-to-many fan-out risk.

**Week 3 boundary:** exploration only. This notebook does **not** create Bronze, Silver, Gold, quarantine, streaming, or Power BI objects.


## Week 3 expected outcome

This notebook produces the Week 3 engineering evidence:

1. Source inventory and file-availability check
2. Physical row counts and distinct business-key counts
3. Schema inspection
4. Null/blank profiling
5. Primary/business-key uniqueness checks
6. Status/category/domain profiling
7. Referential-integrity anti-joins
8. Child-count distributions by `order_id`
9. Quantified unsafe raw-join fan-out demonstration
10. Safe independent child aggregation at `order_id` grain

The CartFlow project defines Week 3 as **profiling, relationships and fan-out tests**. Its expected result is a profile register, anti-join evidence, relationship explanation and quantified fan-out demonstration.


In [ ]:
from pyspark.sql import functions as F

# Use the same CartFlow Volume path already present in your current Week-3 notebook.
# If Catalog Explorer shows your approved CartFlow Volume under a different path,
# change only this variable.
VOLUME_PATH = "/Volumes/cartflow-06/default/cartflow-p06"

print("CartFlow Volume:", VOLUME_PATH)
display(dbutils.fs.ls(VOLUME_PATH))


## 1. Read the five approved batch sources

Approved batch sources:

| File | Format | Grain |
|---|---|---|
| `orders.csv` | CSV | One order lifecycle header |
| `order_items.parquet` | Parquet | One product-seller item line |
| `payments.csv` | CSV | One payment/installment row |
| `reviews.csv` | CSV | One synthetic eligible review |
| `sellers.json` | JSON Lines | One fictional seller master row |

The two `order_status_drop_*.json` files are incremental event inputs reserved for Week 10 and are not processed here.


In [ ]:
orders_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{VOLUME_PATH}/orders.csv")
)

order_items_df = spark.read.parquet(f"{VOLUME_PATH}/order_items.parquet")

payments_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{VOLUME_PATH}/payments.csv")
)

reviews_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{VOLUME_PATH}/reviews.csv")
)

sellers_df = (
    spark.read
    .option("multiLine", False)
    .json(f"{VOLUME_PATH}/sellers.json")
)

orders_df.createOrReplaceTempView("orders")
order_items_df.createOrReplaceTempView("order_items")
payments_df.createOrReplaceTempView("payments")
reviews_df.createOrReplaceTempView("reviews")
sellers_df.createOrReplaceTempView("sellers")

print("All five approved batch sources loaded.")


In [ ]:
source_inventory = [
    ("orders.csv", "CSV", 100000, "order_id", "One order lifecycle header"),
    ("order_items.parquet", "Parquet", 120000, "order_item_id", "One product-seller item line"),
    ("payments.csv", "CSV", 110000, "payment_id", "One payment/installment row"),
    ("reviews.csv", "CSV", 85000, "review_id", "One synthetic eligible review"),
    ("sellers.json", "JSON Lines", 3000, "seller_id", "One fictional seller master row"),
]

display(
    spark.createDataFrame(
        source_inventory,
        ["source_file", "format", "approved_physical_count", "primary_key", "approved_grain"]
    )
)


## 2. Actual source counts and distinct business keys

In [ ]:
count_rows = [
    ("orders", orders_df.count(), orders_df.select("order_id").distinct().count()),
    ("order_items", order_items_df.count(), order_items_df.select("order_item_id").distinct().count()),
    ("payments", payments_df.count(), payments_df.select("payment_id").distinct().count()),
    ("reviews", reviews_df.count(), reviews_df.select("review_id").distinct().count()),
    ("sellers", sellers_df.count(), sellers_df.select("seller_id").distinct().count()),
]

display(
    spark.createDataFrame(
        count_rows,
        ["source", "physical_rows", "distinct_primary_key"]
    )
)


## 3. Schema inspection

In [ ]:
print("ORDERS")
orders_df.printSchema()

print("ORDER ITEMS")
order_items_df.printSchema()

print("PAYMENTS")
payments_df.printSchema()

print("REVIEWS")
reviews_df.printSchema()

print("SELLERS")
sellers_df.printSchema()


## 4. Null / blank profiling

In [ ]:
def null_profile(df, table_name):
    expressions = [
        F.sum(
            F.when(
                F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""),
                1
            ).otherwise(0)
        ).alias(c)
        for c in df.columns
    ]
    return df.select(expressions).withColumn("source", F.lit(table_name))

display(null_profile(orders_df, "orders"))
display(null_profile(order_items_df, "order_items"))
display(null_profile(payments_df, "payments"))
display(null_profile(reviews_df, "reviews"))
display(null_profile(sellers_df, "sellers"))


## 5. Primary/business-key uniqueness

In [ ]:
key_checks = [
    (
        "orders",
        "order_id",
        orders_df.filter(F.col("order_id").isNull()).count(),
        orders_df.count(),
        orders_df.select("order_id").distinct().count()
    ),
    (
        "order_items",
        "order_item_id",
        order_items_df.filter(F.col("order_item_id").isNull()).count(),
        order_items_df.count(),
        order_items_df.select("order_item_id").distinct().count()
    ),
    (
        "payments",
        "payment_id",
        payments_df.filter(F.col("payment_id").isNull()).count(),
        payments_df.count(),
        payments_df.select("payment_id").distinct().count()
    ),
    (
        "reviews",
        "review_id",
        reviews_df.filter(F.col("review_id").isNull()).count(),
        reviews_df.count(),
        reviews_df.select("review_id").distinct().count()
    ),
    (
        "sellers",
        "seller_id",
        sellers_df.filter(F.col("seller_id").isNull()).count(),
        sellers_df.count(),
        sellers_df.select("seller_id").distinct().count()
    ),
]

display(
    spark.createDataFrame(
        key_checks,
        ["source", "key", "null_key_rows", "physical_rows", "distinct_key_rows"]
    )
)


## 6. Domain profiling

In [ ]:
display(orders_df.groupBy("order_status").count().orderBy(F.desc("count")))
display(orders_df.groupBy("customer_segment").count().orderBy(F.desc("count")))

display(order_items_df.groupBy("category_code").count().orderBy(F.desc("count")))

display(payments_df.groupBy("payment_method").count().orderBy(F.desc("count")))

display(reviews_df.groupBy("review_score").count().orderBy("review_score"))

display(sellers_df.groupBy("seller_type").count().orderBy(F.desc("count")))
display(sellers_df.groupBy("service_band").count().orderBy(F.desc("count")))


## 7. Timestamp range profiling

In [ ]:
display(
    orders_df.select(
        F.min("purchase_ts").alias("min_purchase_ts"),
        F.max("purchase_ts").alias("max_purchase_ts"),
        F.min("approval_ts").alias("min_approval_ts"),
        F.max("approval_ts").alias("max_approval_ts"),
        F.min("carrier_handoff_ts").alias("min_carrier_handoff_ts"),
        F.max("carrier_handoff_ts").alias("max_carrier_handoff_ts"),
        F.min("delivered_ts").alias("min_delivered_ts"),
        F.max("delivered_ts").alias("max_delivered_ts")
    )
)

display(
    order_items_df.select(
        F.min("item_created_ts").alias("min_item_created_ts"),
        F.max("item_created_ts").alias("max_item_created_ts")
    )
)

display(
    payments_df.select(
        F.min("payment_ts").alias("min_payment_ts"),
        F.max("payment_ts").alias("max_payment_ts")
    )
)

display(
    reviews_df.select(
        F.min("review_date").alias("min_review_date"),
        F.max("review_date").alias("max_review_date")
    )
)


## 8. Referential-integrity anti-joins

In [ ]:
items_missing_orders = order_items_df.join(
    orders_df.select("order_id").distinct(),
    "order_id",
    "left_anti"
)

payments_missing_orders = payments_df.join(
    orders_df.select("order_id").distinct(),
    "order_id",
    "left_anti"
)

reviews_missing_orders = reviews_df.join(
    orders_df.select("order_id").distinct(),
    "order_id",
    "left_anti"
)

items_missing_sellers = order_items_df.join(
    sellers_df.select("seller_id").distinct(),
    "seller_id",
    "left_anti"
)

anti_join_results = [
    ("order_items -> orders", items_missing_orders.count()),
    ("payments -> orders", payments_missing_orders.count()),
    ("reviews -> orders", reviews_missing_orders.count()),
    ("order_items -> sellers", items_missing_sellers.count()),
]

display(
    spark.createDataFrame(
        anti_join_results,
        ["relationship", "unmatched_child_rows"]
    )
)


## 9. Child-count distributions by `order_id`

In [ ]:
item_counts = (
    order_items_df
    .groupBy("order_id")
    .count()
    .withColumnRenamed("count", "item_count")
)

payment_counts = (
    payments_df
    .groupBy("order_id")
    .count()
    .withColumnRenamed("count", "payment_count")
)

review_counts = (
    reviews_df
    .groupBy("order_id")
    .count()
    .withColumnRenamed("count", "review_count")
)

display(item_counts.groupBy("item_count").count().orderBy("item_count"))
display(payment_counts.groupBy("payment_count").count().orderBy("payment_count"))
display(review_counts.groupBy("review_count").count().orderBy("review_count"))


## 10. Quantified one-to-many fan-out demonstration

CartFlow's central engineering risk is joining multiple child tables directly at their physical grain.

The safe approach is:

**items → aggregate to `order_id`**

**payments → aggregate to `order_id`**

then join the two order-level summaries.

The unsafe approach below joins item rows directly to payment rows, which can multiply both item and payment amounts when an order has multiple child rows.


In [ ]:
# SAFE: independently aggregate children to order grain.
items_by_order = (
    order_items_df
    .groupBy("order_id")
    .agg(
        F.sum("item_price").alias("item_value"),
        F.sum("freight_value").alias("freight_value"),
        F.count("*").alias("item_rows")
    )
)

payments_by_order = (
    payments_df
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("payment_value"),
        F.count("*").alias("payment_rows")
    )
)

safe_totals = (
    items_by_order
    .join(payments_by_order, "order_id", "outer")
    .agg(
        F.sum("item_value").alias("safe_item_value"),
        F.sum("payment_value").alias("safe_payment_value"),
        F.sum("item_rows").alias("item_rows"),
        F.sum("payment_rows").alias("payment_rows")
    )
)

# UNSAFE: raw item rows joined to raw payment rows.
unsafe = (
    orders_df.select("order_id")
    .join(
        order_items_df.select("order_id", "item_price"),
        "order_id",
        "inner"
    )
    .join(
        payments_df.select("order_id", "payment_value"),
        "order_id",
        "inner"
    )
)

unsafe_totals = unsafe.agg(
    F.count("*").alias("unsafe_join_rows"),
    F.sum("item_price").alias("unsafe_item_value"),
    F.sum("payment_value").alias("unsafe_payment_value")
)

display(safe_totals)
display(unsafe_totals)


In [ ]:
comparison = (
    safe_totals
    .crossJoin(unsafe_totals)
    .select(
        "safe_item_value",
        "unsafe_item_value",
        "safe_payment_value",
        "unsafe_payment_value",
        "item_rows",
        "payment_rows",
        "unsafe_join_rows"
    )
    .withColumn(
        "item_value_inflation",
        F.col("unsafe_item_value") - F.col("safe_item_value")
    )
    .withColumn(
        "payment_value_inflation",
        F.col("unsafe_payment_value") - F.col("safe_payment_value")
    )
)

display(comparison)


## 11. CartFlow engineering conclusion

Use the following guardrail in later notebooks:

**Aggregate item and payment children independently to `order_id`, validate complete coverage, reconcile money within INR 0.05, then join each summary exactly once to build `fact_order`. Never raw-join items, payments and events together.**

This notebook intentionally stops before Bronze. Bronze begins in Week 4.


In [ ]:
week3_evidence = [
    ("source_inventory", "Five approved batch sources identified"),
    ("counts_and_grain", "Physical rows and distinct business keys calculated"),
    ("schema_profile", "Source schemas inspected"),
    ("null_profile", "Null/blank counts calculated"),
    ("key_uniqueness", "Primary/business-key uniqueness tested"),
    ("domain_profile", "Status/category/payment/review/seller domains profiled"),
    ("anti_joins", "Child-to-order and item-to-seller unmatched rows quantified"),
    ("child_distributions", "Child-count distributions calculated by order_id"),
    ("fanout_demo", "Unsafe raw join compared with independent child aggregation"),
]

display(
    spark.createDataFrame(
        week3_evidence,
        ["evidence_item", "what_to_prove"]
    )
)


## Week 3 exit checklist

- [ ] Execute every cell in Databricks.
- [ ] Keep the real outputs that prove the Week 3 work.
- [ ] Record genuine findings rather than inventing PASS/FAIL results.
- [ ] Update `docs/pipeline_walkthrough.md` with the fan-out guardrail.
- [ ] Update `weekly_logs/week03_log.md`.
- [ ] Save genuine Week-3 evidence.
- [ ] Do **not** create Bronze tables in this notebook.
